# 01 - Data Preprocessing

**Purpose**: Clean, transform, and prepare data for clustering analysis

**Steps**:
1. Load raw data from previous notebook
2. Handle missing values (imputation)
3. Calculate CAGR (Compound Annual Growth Rate) features
4. Feature scaling and normalization
5. Create analysis-ready datasets

**Outputs**:
- `df_features`: Cleaned and engineered features
- `df_latest`: Most recent snapshot
- `df_all`: Time-series data

---

## 1. Setup & Load Previous State

In [ ]:
# Import utilities
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if 'notebooks' in str(Path.cwd()) else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from notebook_utils import setup_notebook, display_dataframe_summary
from src._02_preprocessing import data_cleaner

%matplotlib inline
print("✓ Imports loaded")

In [ ]:
# Load state from previous notebook
cfg, state = setup_notebook(
    title="Data Preprocessing & Feature Engineering",
    market="germany"
)

# Load saved configuration
config = state.load('config')
market = config['market']

print(f"\n✓ Loaded configuration for market: {market}")

## 2. Configure Preprocessing Parameters

**Adjust these parameters as needed:**

In [ ]:
# ============================================================================
# PREPROCESSING CONFIGURATION - Edit these parameters as needed
# ============================================================================

# Imputation settings
IMPUTE_ENABLED = True           # Enable/disable imputation
IMPUTE_METHOD = 'median'        # Options: 'mean', 'median', 'zero'
IMPUTE_THRESHOLD = 0.5          # Drop columns with >50% missing values

# CAGR calculation
SMOOTH_STATIC = False           # Apply smoothing to static features
CAGR_YEARS = 3                  # Number of years for CAGR calculation

# Display settings
print("📋 Preprocessing Configuration:")
print("=" * 80)
print(f"Imputation Enabled:    {IMPUTE_ENABLED}")
print(f"Imputation Method:     {IMPUTE_METHOD}")
print(f"Imputation Threshold:  {IMPUTE_THRESHOLD}")
print(f"Smooth Static:         {SMOOTH_STATIC}")
print(f"CAGR Years:            {CAGR_YEARS}")
print("=" * 80)

## 3. Run Preprocessing Pipeline

In [ ]:
# Run preprocessing
print("\n🔄 Running preprocessing pipeline...\n")

input_dir = config['input_dir']

df_features = data_cleaner.run_preprocessing(
    input_dir=input_dir,
    market=market,
    impute=IMPUTE_ENABLED,
    impute_method=IMPUTE_METHOD,
    impute_threshold=IMPUTE_THRESHOLD,
    smooth_static=SMOOTH_STATIC,
    cagr_years=CAGR_YEARS
)

print("\n✓ Preprocessing complete!")
display_dataframe_summary(df_features, "Processed Features")

In [ ]:
# Preview data
print("\n📊 Preview of processed data:")
display(df_features.head(10))

## 4. Feature Analysis

In [ ]:
# Identify feature types
numeric_cols = df_features.select_dtypes(include=[np.number]).columns.tolist()

# Exclude ID and year columns
exclude_cols = ['company_id', 'year', 'cluster', 'label']
feature_cols = [c for c in numeric_cols if c not in exclude_cols]

# Categorize features
static_features = [f for f in feature_cols if not f.endswith('_cagr')]
dynamic_features = [f for f in feature_cols if f.endswith('_cagr')]

print(f"\n📊 Feature Overview:")
print("=" * 80)
print(f"Total Features:        {len(feature_cols)}")
print(f"Static Features:       {len(static_features)}")
print(f"Dynamic Features:      {len(dynamic_features)}")
print("=" * 80)

print(f"\nStatic Features ({len(static_features)}):")
for f in static_features[:10]:
    print(f"  - {f}")
if len(static_features) > 10:
    print(f"  ... and {len(static_features) - 10} more")

print(f"\nDynamic Features ({len(dynamic_features)}):")
for f in dynamic_features[:10]:
    print(f"  - {f}")
if len(dynamic_features) > 10:
    print(f"  ... and {len(dynamic_features) - 10} more")

## 5. Data Quality Visualization

In [ ]:
# Missing values heatmap
missing_data = df_features[feature_cols].isnull()
missing_pct = (missing_data.sum() / len(df_features) * 100).sort_values(ascending=False)

if missing_pct.sum() > 0:
    # Show top 20 features with missing values
    top_missing = missing_pct[missing_pct > 0].head(20)
    
    plt.figure(figsize=(12, 6))
    top_missing.plot(kind='barh', color='salmon')
    plt.xlabel('Missing %')
    plt.title('Top 20 Features with Missing Values')
    plt.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("✓ No missing values in features!")

In [ ]:
# Feature distribution statistics
feature_stats = df_features[feature_cols].describe().T

print("\n📊 Feature Statistics (First 10):")
display(feature_stats.head(10))

In [ ]:
# Visualize distributions of selected features
sample_features = feature_cols[:6]  # First 6 features

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for idx, feature in enumerate(sample_features):
    ax = axes[idx]
    df_features[feature].hist(bins=50, ax=ax, color='steelblue', alpha=0.7)
    ax.set_title(feature, fontsize=10)
    ax.set_xlabel('Value')
    ax.set_ylabel('Frequency')
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.suptitle('Feature Distributions (Sample)', y=1.02, fontsize=14, fontweight='bold')
plt.show()

## 6. Prepare Time-Series Data

In [ ]:
# Create time-split datasets
print("\n🔄 Preparing time-series datasets...\n")

df_all, df_latest = data_cleaner.prepare_time_data(df_features, market)

print("\n✓ Time-series preparation complete!\n")
display_dataframe_summary(df_all, "All Time Periods")
display_dataframe_summary(df_latest, "Latest Snapshot")

In [ ]:
# Visualize temporal coverage
if 'year' in df_all.columns:
    year_counts = df_all['year'].value_counts().sort_index()
    
    plt.figure(figsize=(12, 5))
    year_counts.plot(kind='bar', color='steelblue', alpha=0.7)
    plt.xlabel('Year')
    plt.ylabel('Number of Companies')
    plt.title('Temporal Coverage: Companies per Year')
    plt.xticks(rotation=45)
    plt.grid(axis='y', alpha=0.3)
    
    # Add value labels
    for i, v in enumerate(year_counts.values):
        plt.text(i, v + 10, str(v), ha='center', va='bottom')
    
    plt.tight_layout()
    plt.show()
    
    print(f"\nYear Range: {year_counts.index.min()} - {year_counts.index.max()}")
    print(f"Total Years: {len(year_counts)}")

## 7. Feature Correlation Analysis

In [ ]:
# Compute correlation matrix for top features
top_features = feature_cols[:15]  # Top 15 features
corr_matrix = df_latest[top_features].corr()

# Plot correlation heatmap
plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', 
            center=0, square=True, linewidths=0.5, 
            cbar_kws={'label': 'Correlation'})
plt.title('Feature Correlation Matrix (Top 15 Features)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Find highly correlated pairs
high_corr_pairs = []
for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        if abs(corr_matrix.iloc[i, j]) > 0.8:
            high_corr_pairs.append((
                corr_matrix.columns[i],
                corr_matrix.columns[j],
                corr_matrix.iloc[i, j]
            ))

if high_corr_pairs:
    print("\n⚠️  Highly Correlated Feature Pairs (|r| > 0.8):")
    for feat1, feat2, corr in high_corr_pairs:
        print(f"  {feat1} <-> {feat2}: {corr:.3f}")
else:
    print("\n✓ No highly correlated feature pairs found")

## 8. Save Preprocessed Data

In [ ]:
# Save to state for next notebooks
state.save('df_features', df_features, metadata={'shape': df_features.shape})
state.save('df_all', df_all, metadata={'shape': df_all.shape})
state.save('df_latest', df_latest, metadata={'shape': df_latest.shape})

# Save feature lists
state.save('feature_cols', feature_cols)
state.save('static_features', static_features)
state.save('dynamic_features', dynamic_features)

print("\n✓ All preprocessed data saved to state")

## 9. Summary

In [ ]:
print("\n" + "=" * 80)
print("  ✓ PREPROCESSING COMPLETE")
print("=" * 80)

print(f"\n📊 Output Summary:")
print(f"  df_features:     {df_features.shape[0]:,} rows × {df_features.shape[1]:,} columns")
print(f"  df_all:          {df_all.shape[0]:,} rows × {df_all.shape[1]:,} columns")
print(f"  df_latest:       {df_latest.shape[0]:,} rows × {df_latest.shape[1]:,} columns")
print(f"\n  Total Features:  {len(feature_cols)}")
print(f"  Static:          {len(static_features)}")
print(f"  Dynamic:         {len(dynamic_features)}")

print("\n📝 Next Steps:")
print("  → Open notebook: 02_KMeans_Clustering.ipynb")
print("  → Or explore: 03_Hierarchical_Clustering.ipynb")
print("  → Or explore: 04_DBSCAN_Clustering.ipynb")
print("=" * 80)